In [55]:
import wandb
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from scipy.stats import ks_2samp, chi2_contingency
import random
import os
import sys
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
sys.path.append(r'C:\Users\Gabri\OneDrive\Desktop\Projeto Kaggle IA')

from src.data_cleaning import pipeline_clean_date
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from src.feature_selection import select_top_features

# Configurar semente para reprodutibilidade
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [56]:
wandb.login()

True

<h2>Hyperparametrs in Config</4>

<h4><b>Test size:</b> 20% of the data is separated for testing and 80% for training. This is a classic machine learning ratio. A test smaller than this might not accurately represent reality, and a larger test would reduce the training data, as there was already little data with a positive biopsy, which could harm the model.</h4>
<h4><b>hidden_sizes:</b> This is used for the first layer to learn the patterns, and then filter them, taking what is most important. I tested other settings such as [128, 64,32] (more neurons) and [16, 8] (fewer neurons). The larger layers caused overfitting (the model memorized the training data and performed poorly on the test). The smaller layers were too weak to learn complex patterns of cervical cancer. For tabular problems, 1 to 3 layers are sufficient.
<h4><b>Dropout:</b> Randomly turns off a percentage of neurons according to your choice, adjusted precisely to avoid overfitting, forcing the model not to memorize specific patterns. I tested larger values ​​and there were too many missing values ​​for it to adapt, and even fewer to turn it into a pattern repeater.</h4>
<h4><b>Learning size:</b> The amount of speed the machine has to learn; I tried put more slow learning_rate, where it demonstrated good performance.</h4>

In [67]:
config = {
    "data": {
        "raw_path": "data/raw/kag_risk_factors_cervical_cancer.csv",
        "test_size": 0.2,
        "random_state": 42,
        "target_col": "Biopsy",
        "missing_threshold": 0.5,
        "imputation_strategy": "median"
    },
    "model": {
        "hidden_sizes": [64,32],
        "dropout": 0.2,
        "learning_rate": 0.0005,
        "batch_size": 32,
        "epochs": 100,
        "early_stopping_patience": 10
    }
}

<h3>Raw Data Artifact for Registration in Wandb</h3>

In [58]:
path = r"../data/raw/kag_risk_factors_cervical_cancer.csv"
df_raw = pd.read_csv(path)

wandb.init(project="cervical-cancer-analysis", job_type="load_raw", name="load_raw")
artifact = wandb.Artifact("raw_data", type="dataset")

temp_path = r"C:\Users\Gabri\OneDrive\Desktop\temp_raw.csv"

df_raw.to_csv(temp_path, index=False)
artifact.add_file(temp_path)
wandb.log_artifact(artifact)

wandb.summary["rows"] = len(df_raw)
wandb.summary["columns"] = list(df_raw.columns)
wandb.finish()

print("Artefato raw_data salvo no W&B")



rows,858


Artefato raw_data salvo no W&B


<h3>Clean Data Artifact for Registration in Wandb</h3>

In [59]:

wandb.init(project="cervical-cancer-analysis", job_type="clean_data", name="clean_data")
artifact = wandb.Artifact("clean_data", type="dataset")
temp_path = r"C:\Users\Gabri\OneDrive\Desktop\temp_clean.csv"
df_clean = pipeline_clean_date(df_raw)
df_clean.to_csv(temp_path, index=False)
artifact.add_file(temp_path)
wandb.log_artifact(artifact)
wandb.summary["rows"] = len(df_clean)
wandb.finish()

(858, 36)
Columns removed: 2
Rows removed: 98
Numbers of Duplicates Removed 19
(741, 34)


rows,741


<h3><b>Split:</b> is used to divide the data into test and training sets, maintaining the proportion of classes (stratify=y). It serves precisely to have two random datasets so that the model does not have overfitting and to know if it is predicting the data correctly. We can adjust the proportion between the amount of data for testing and training.</h3>
<h3><b>Compare:</b> Comparing the results of the data division into test and training sets and verifying if the split was done correctly. KS (Numerical data, verifying if the samples came from the same distribution) and Chi2 (Categorical data) are static tests to compare the distribution and validate the split.</h3>
<h3><b>Applying Feature Select after Split</b></h3>
<h3><b>Saving the Train and Test data in Wandb</b></h3>

In [60]:
def split_train_test(df, target_col, test_size=0.2, random_state=42):
    print(df.columns)
    print(target_col)
    X = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y #Stratify, mantem a proporção de quantidade de casos positivos e negativos, necessário devido ao desbalanceamento de classes
    )
    train_df = pd.concat([X_train, y_train], axis=1)
    test_df = pd.concat([X_test, y_test], axis=1)
    return train_df, test_df

def compare_distributions(train_df, test_df, columns):
    results = {}
    for col in columns:
        train_vals = train_df[col].dropna()
        test_vals = test_df[col].dropna()
        if train_df[col].dtype in ['int64', 'float64']:
            ks_stat, p_value = ks_2samp(train_vals, test_vals)
            results[col] = {'test': 'KS', 'statistic': ks_stat, 'p_value': p_value}
        else:
            train_counts = train_vals.value_counts(normalize=True)
            test_counts = test_vals.value_counts(normalize=True)
            all_cats = sorted(set(train_counts.index).union(set(test_counts.index)))
            train_probs = [train_counts.get(cat, 0) for cat in all_cats]
            test_probs = [test_counts.get(cat, 0) for cat in all_cats]
            chi2, p_value, _, _ = chi2_contingency([train_probs, test_probs])
            results[col] = {'test': 'Chi2', 'statistic': chi2, 'p_value': p_value}
    return results


#df_final = pipeline_feacture_selection(df_clean)

#print(f"Shape: {df_final.shape}")
#print(f"Colunas: {df_final.columns.tolist()}")
#print(f"Tem Biopsy? {'Biopsy' in df_final.columns}")

train_df, test_df = split_train_test(df_clean, 
                                     target_col=config['data']['target_col'],
                                     test_size=config['data']['test_size'],
                                     random_state=config['data']['random_state'])

#Feacture Select
print(f"Treino: {len(train_df)} amostras")
print(f"Teste:  {len(test_df)} amostras")
print(train_df.shape)

X_train = train_df.drop(columns=['Biopsy'])
y_train = train_df['Biopsy']
top_cols = select_top_features(X_train, y_train, n_top=13 ) #Numbers of select feacture, 13>12>15>10
train_df = train_df[top_cols + ['Biopsy']]
test_df = test_df[top_cols + ['Biopsy']]


print(f"Treino após FS: {train_df.shape}")
print(f"Teste após FS: {test_df.shape}")
print(f"Features: {top_cols}")

feature_cols = [c for c in train_df.columns if c != config['data']['target_col']]
comp_results = compare_distributions(train_df, test_df, feature_cols)

wandb.init(project="cervical-cancer-analysis", job_type="split_data", name="split_data")
train_artifact = wandb.Artifact("train_data", type="dataset")
path_temp_train = r"../temp_train.csv"
train_df.to_csv(path_temp_train, index=False)
train_artifact.add_file(path_temp_train)
wandb.log_artifact(train_artifact)

test_artifact = wandb.Artifact("test_data", type="dataset")
path_temp_test = r"../temp_test.csv"
test_df.to_csv(path_temp_test, index=False)
test_artifact.add_file(path_temp_test)
wandb.log_artifact(test_artifact)

comp_df = pd.DataFrame(comp_results).T
comp_table = wandb.Table(dataframe=comp_df)
wandb.log({"distribution_comparison": comp_table})

wandb.summary["train_size"] = len(train_df)
wandb.summary["test_size"] = len(test_df)

wandb.finish()
print("Artefatos de treino e teste salvos com sucesso.")

Index(['Age', 'Number of sexual partners', 'First sexual intercourse',
       'Num of pregnancies', 'Smokes', 'Smokes (years)', 'Smokes (packs/year)',
       'Hormonal Contraceptives', 'Hormonal Contraceptives (years)', 'IUD',
       'IUD (years)', 'STDs', 'STDs (number)', 'STDs:condylomatosis',
       'STDs:cervical condylomatosis', 'STDs:vaginal condylomatosis',
       'STDs:vulvo-perineal condylomatosis', 'STDs:syphilis',
       'STDs:pelvic inflammatory disease', 'STDs:genital herpes',
       'STDs:molluscum contagiosum', 'STDs:AIDS', 'STDs:HIV',
       'STDs:Hepatitis B', 'STDs:HPV', 'STDs: Number of diagnosis',
       'Dx:Cancer', 'Dx:CIN', 'Dx:HPV', 'Dx', 'Hinselmann', 'Schiller',
       'Citology', 'Biopsy'],
      dtype='object')
Biopsy
Treino: 592 amostras
Teste:  149 amostras
(592, 34)


c:\Users\Gabri\OneDrive\Desktop\Projeto Kaggle IA\.venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
c:\Users\Gabri\OneDrive\Desktop\Projeto Kaggle IA\.venv\Lib\site-packages\statsmodels\regression\linear_model.py:1784: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.uncentered_tss


Treino após FS: (592, 14)
Teste após FS: (149, 14)
Features: ['Schiller', 'Hinselmann', 'Dx:CIN', 'Citology', 'Hormonal Contraceptives (years)', 'Num of pregnancies', 'Number of sexual partners', 'Dx:Cancer', 'Dx', 'Dx:HPV', 'Smokes', 'IUD (years)', 'Hormonal Contraceptives']


test_size,149
train_size,592


Artefatos de treino e teste salvos com sucesso.


<h3><b>(MLP) REDE NEURAL MULTILAYER PERCEPTRON </b></h3>
<h4>
<b>ReLU:</b> é a que eu usei nas camadas ocultas porque é simples, rápida e funciona muito bem na prática. Ela ajuda o modelo a aprender melhor e evita o problema de um gradiente muito pequeno, mas pode ocasionar neuronios mortos, entretanto não sofre de vanishing gradient ( modelo para de aprender)</h4>
<h4><b>Tanh</b> Varia entre [-1,1], sofre vanishing gradient, pela maneira que ele ordena os dados, a função fica ou muito alta, ou muito baixa, ficando constante, e acaba não evoluindo o modelo</h4>
<h4><b>Sigmoid:</b> gera valores entre 0 e 1, então faz sentido usar na saída quando o problema é de classificação binária ( Como nosso caso ), entretanto ainda sofre vanishing gradient</h4>
<h4><b>LeakyReLU:</b> é muito parecido com ReLU, entretanto tenta corrigir seus defeitos, mas não é usado porque ReLU já funciona bem na maioria dos casos, e com menos complexidade
</h4>

In [61]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_sizes, output_dim=1, dropout=0.2):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU()) #ReLU
            layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

<h4>Accuracy → proporção total de acertos (Problema Aparente devido a desbalanceamento das calsses)</h4>
<h4>Precision → qualidade das previsões positivas</h4>
<h4>Recall → capacidade de detectar casos positivosh4>
<h4>F1-score → equilíbrio entre precision e recall</h4>
<h4>AUC-ROC → capacidade geral de separação entre classes</h4>


In [70]:
def prepare_dataloaders(train_df, test_df, target_col, batch_size):
    #Separate the target column and transform it through reshaping
    X_train = train_df.drop(columns=[target_col]).values.astype(np.float32)
    y_train = train_df[target_col].values.astype(np.float32).reshape(-1, 1)
    X_test = test_df.drop(columns=[target_col]).values.astype(np.float32)
    y_test = test_df[target_col].values.astype(np.float32).reshape(-1, 1)
    
    #Data normalization
    scaler = RobustScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    #Convert numpy arrays to PyTorch tensors
    train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    test_dataset = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))
    
    #Number of samples per batch
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, test_loader, scaler

def train_model(config, train_loader, test_loader, input_dim):
    #Selecting the CPU to use
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    #Create the template with the configuration settings
    model = MLP(input_dim, config['model']['hidden_sizes'], dropout=config['model']['dropout']).to(device)

    #Apply Sigmoid, transforming the outputs into probability [0,1]
    criterion = nn.BCEWithLogitsLoss() #combina sigmoid + BCE
    #Optimization through Learning Rate
    optimizer = optim.Adam(model.parameters(), lr=config['model']['learning_rate'])
    
    #Early Stopping
    best_loss = float('inf') #Para comparação com val_loss
    patience = config['model']['early_stopping_patience']
    counter = 0

    wandb.watch(model, log="all")

    for epoch in range(config['model']['epochs']):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0) 
        #Train_loss is used to see how much the model is improving over time; if there are no significant changes in the decrease in value, it means overlifting. 
        train_loss /= len(train_loader.dataset) 

        model.eval() 
        val_loss = 0.0
        correct = 0
        all_preds = []   
        all_labels = []  
        all_probs = []

        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                output = model(X_batch)
                loss = criterion(output, y_batch)
                val_loss += loss.item() * X_batch.size(0) 
                
                #Use probability to define the cases; a probability greater than 20% is positive, making it more sensitive, but it also generates more false positives.
                threshold = 0.2
                pred = (torch.sigmoid(output) > threshold).float()
                probs = torch.sigmoid(output)
                correct += (pred == y_batch).sum().item()

                all_probs.extend(probs.cpu().numpy().flatten())
                all_preds.extend(pred.cpu().numpy())      #F1
                all_labels.extend(y_batch.cpu().numpy())  #F1
        try:
            #The model's ability to distinguish between two classes, positive and negative, error handling is due to the chance of having only one class
            auc_roc = roc_auc_score(all_labels, all_probs)
        except:
            auc_roc = 0.5  

        #Val_Loss calculates the model's data and validation errors; a low val_loss indicates good model generalization, a high val_loss indicates overfitting; a significant difference from train_loss indicates a problem.
        val_loss /= len(test_loader.dataset) 
        #Model accuracy, the number of total correct predictions in relation to the entire dataset; the model here is biased towards always looking at the positives, which is a problem and a more worrying statistic.
        acc = correct / len(test_loader.dataset)
        #F1 Balancing Precision and Recall (A bit redundant to keep but I'll leave it), the higher the F1, the better it's performing its function.
        f1 = f1_score(all_labels, all_preds, zero_division=0) #New Metrics
        #Number of actual predictions that the model identified, of positive patients
        recall = recall_score(all_labels, all_preds, zero_division=0)      
        #Measures the quality of positive predictions made by the model, where it analyzes all positive cases.
        precision = precision_score(all_labels, all_preds, zero_division=0) 

        #Saving Metrics
        wandb.log({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_acc": acc,
                   "val_f1": f1, "val_auc": auc_roc,"val_recall": recall,"val_precision": precision}) #New metrics

        
        #Early Stopping configuration: if there is no improvement in 10 different epochs, the program will simply stop, also defined in the configuration.
        #Saves the best model through this code block.
        if val_loss < best_loss:
            best_loss = val_loss
            counter = 0
            torch.save(model.state_dict(), "best_model.pt")

        else:
            counter += 1
            if counter >= patience:
                print(f"Early stopping triggered at epoch {epoch}")
                break

    model_artifact = wandb.Artifact("trained_model", type="model")
    model_artifact.add_file("best_model.pt")
    wandb.log_artifact(model_artifact)

    #Matrix confusion configuration, I configured it incorrectly and initially it sent too many images to Wandb.
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
        
    wandb.log({f"confusion_matrix - epoch {epoch}": wandb.Image(plt)})
    plt.close()
    
    model.load_state_dict(torch.load("best_model.pt"))
    return model

wandb.init(project="cervical-cancer-analysis", job_type="train", name="64_32_dropout0.2", config=config)

train_df = pd.read_csv(path_temp_train)
test_df = pd.read_csv(path_temp_test)

train_loader, test_loader, scaler = prepare_dataloaders(
    train_df, test_df, config['data']['target_col'], config['model']['batch_size']
)
input_dim = train_loader.dataset.tensors[0].shape[1]

model = train_model(config, train_loader, test_loader, input_dim)

wandb.finish()
print("Treinamento concluído e melhor modelo salvo no W&B.")

Early stopping triggered at epoch 43


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train_loss,█▇▆▅▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▁▂▃▄▅██████████████████████████████████
val_auc,▂▁▁▁▁▂▃▄▄▄▅▆▆▇██████████████████████████
val_f1,▂▂▁▁▁▁▃▂▂▂▇▆▆▆█▇▇▇████████████████▇█▇███
val_loss,█▇▆▅▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_precision,▁▁▁▁▁▁▄████▇▆▆▆▆▆▆▆▆▆▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅
val_recall,██▇▅▄▂▂▁▁▁▄▄▄▄▅▅▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▆▇▆▇▇▇
epoch,43
train_loss,0.08151
val_acc,0.95302


Treinamento concluído e melhor modelo salvo no W&B.


<h2><b>Discurssão Final<b></h2>
<h3>O modelo criado, apresenta desempenhos razoáveis, considerando o forte desbalanceamento entre as classes, gerava resultados problemáticos, priorizei identificar a maioria dos casos positivos, mesmo que isso gerasse custos em uma aplicação real, imaginei que era melhor falsos positivos, do que poucos positivos de fato, o modelo atende ao problema proposto, mas devido a baixa quantidade de casos positivos no totais, acabou gerando problemas, acho que teve um pequeno overfitting, devido as quedas do AUC_ROC, além de eu atribuir importância demais ao tresholder, onde o modelo está variando muito em relação a ele.</h3>
<h3>Acabei fazendo as comparações manualmente, uma melhora atual seria adicionar cross-validation, e técnicas para desbalanceamento do modelo, talvez um data_set com mais casos positivos ajudasse, mas nem sempre em um cenário real eu terei as informações como necessito.</h3>